This notebook has code that was used to test differrent models and different model structures

Also contains code to test the periodicity of the sine and cosine periodicity vs. the target variable using fast fourier transforms

In [ ]:
import pandas as pd
data2 = '/pathtodata/Data2/' # Path to a folder containing seperated data for each geohash
freesupplydr5rus = '/pathtodata/combined_per_minute_supply.csv' # Path to freesupply data

In [ ]:
import os
# Initialize an empty dictionary to store the DataFrames
geohash_dfs = {}

# Loop through the files in the directory
for filename in os.listdir(data2):
    if filename.endswith(".csv"):
        # Extract the geohash from the filename (assuming the filename is in the format 'geohash.csv')
        geohash = filename.split('.')[0]

        # Load the CSV file into a DataFrame
        df = pd.read_csv(os.path.join(data2, filename))

        # Add the DataFrame to the dictionary
        geohash_dfs[geohash] = df

# Verify the loaded DataFrames
for geohash, df in geohash_dfs.items():
    print(f"Geohash: {geohash}")
    print(df.head())  # Display the first few rows of each DataFrame
    print("\n")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
## Code to plot the frequency vs. target variable
fft = tf.signal.rfft(df['time_to_next_ride'])
f_per_dataset = np.arange(0, len(fft))

n_samples_h = len(df['time_to_next_ride'])
hours_per_year = 24*365.2524
years_per_dataset = n_samples_h/(hours_per_year)

f_per_year = f_per_dataset/years_per_dataset
plt.step(f_per_year, np.abs(fft))
plt.xscale('log')
plt.ylim(0, 6000000)
plt.xlim([0.1, max(plt.xlim())])
plt.xticks([1, 365.2524], labels=['1/Year', '1/day'])
_ = plt.xlabel('Frequency (log scale)')

## to describe the major statistics of the data feautures
df.describe().transpose()

Test for an LSTM model for the 'dr5rus' geohash data

In [ ]:
##Setup the data frame
test_df = geohash_dfs['dr5rus'].copy()
trialdf = test_df[['OFFER_DATE','time_of_day','day_of_week','month','temperature','precipitation']]
dr5rus_df = trialdf.copy()

dr5rus_df = dr5rus_df.sort_values(by='OFFER_DATE')

# Convert 'OFFER_DATE' to datetime objects
dr5rus_df['OFFER_DATE'] = pd.to_datetime(dr5rus_df['OFFER_DATE'])

# Shift the OFFER_DATE column to create the next ride's time
dr5rus_df['next_OFFER_DATE'] = dr5rus_df['OFFER_DATE'].shift(-1)

# Calculate the time difference to the next ride
dr5rus_df['time_to_next_ride'] = (dr5rus_df['next_OFFER_DATE'] - dr5rus_df['OFFER_DATE']).dt.total_seconds()

# Drop the temporary column next_OFFER_DATE
dr5rus_df.drop(columns=['next_OFFER_DATE'], inplace=True)
dr5rus_df.dropna(subset=['time_to_next_ride'], inplace=True)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split

In [ ]:
# Select the features and target
features = dr5rus_df[['time_of_day', 'day_of_week', 'month', 'temperature', 'precipitation']]
target = dr5rus_df['time_to_next_ride']

# Normalize the features
feature_scaler = MinMaxScaler()
features_scaled = feature_scaler.fit_transform(features)

# Normalize the target
target_scaler = MinMaxScaler()
target_scaled = target_scaler.fit_transform(target.values.reshape(-1, 1))

# Reshape the data to fit the LSTM input requirements
time_steps = 1  # Since each row is a separate time step
X = np.reshape(features_scaled, (features_scaled.shape[0], time_steps, features_scaled.shape[1]))
y = target_scaled

print("Preprocessed data shape:", X.shape)
print("Target shape:", y.shape)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

In [ ]:
# Build the LSTM model
model = Sequential()
model.add(LSTM(64, activation='relu', input_shape=(time_steps, features_scaled.shape[1])))
model.add(Dense(32, activation='sigmoid'))
model.add(Dense(16, activation='sigmoid'))
model.add(Dense(1, activation='sigmoid'))

# Print the model summary
model.summary()
model.compile(optimizer='adam', loss='mse')

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=1)

In [ ]:
# Evaluate the model
loss = model.evaluate(X_test, y_test, verbose=1)
print(f'Test Loss: {loss}')

Test for finding the best model structure: considering different layer structures, node numbers, optimizers, dropout layers, and convolutional layers

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras_tuner import RandomSearch
# Reshape data for Conv1D layer
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

def build_model(hp):
    model = keras.Sequential()

    # Convolutional layer (optional)
    if hp.Boolean('conv_layer'):
        model.add(keras.layers.Conv1D(
            filters=hp.Int('conv_filters', min_value=8, max_value=128, step=16),
            kernel_size=hp.Choice('conv_kernel_size', values=[3, 5]),
            activation='relu',
            input_shape=(X_train.shape[1], 1)
        ))
        model.add(keras.layers.MaxPooling1D(pool_size=2))
        model.add(keras.layers.Flatten())
    else:
        model.add(keras.layers.Flatten(input_shape=(X_train.shape[1], 1)))

    # Adding multiple dense layers
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(keras.layers.Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=256, step=32),
            activation=hp.Choice('activation', ['relu', 'tanh', 'sigmoid'])
        ))
        if hp.Boolean(f'dropout_{i}'):
            model.add(keras.layers.Dropout(hp.Float(f'dropout_rate_{i}', 0.2, 0.5, step=0.1)))

    # Output layer
    model.add(keras.layers.Dense(1))  # Output layer for regression

    # Compile the model
    model.compile(
        optimizer=hp.Choice('optimizer', ['adam', 'rmsprop', 'sgd']),
        loss='mse',
        metrics=['mae']
    )

    return model

In [ ]:
tuner = RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=25,  # You can increase the number of trials for a more thorough search
    executions_per_trial=2,  # Each trial will run 3 times to average the performance
    directory='my_dir',
    project_name='taxi_time_to_next_ride'
)

# Run the hyperparameter search
tuner.search(X_train, y_train, epochs=30, validation_data=(X_val, y_val))

# Retrieve the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Train the best model
model = tuner.hypermodel.build(best_hps)
model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val))

In [ ]:
model.save('/pathtosave/dr5rustuner1-holiday.keras')
print("Model saved successfully.")